# LAM → OpenAvatarChat (OAC) avatar bake — Colab (free T4)

Bake **one** avatar from **one** portrait with [aigc3d/LAM](https://github.com/aigc3d/LAM) and export the
**OAC zip** (`skin.glb`, `offset.ply`, `animation.glb`, `vertex_order.json`) that
`ints-head-gs` loads. **Not** `h5_render_data.zip`.

**Before anything:** `Runtime ▸ Change runtime type ▸ T4 GPU`. Then run cells 1 → 7 top to bottom.

### Why a conda env (the Python-3.10 problem)
LAM's OAC export builds `skin.glb` with the **FBX SDK**, shipped **only as a cp310 wheel** → it needs
**Python 3.10**. Today's Colab is **Python 3.12**, and condacolab installs its own recent build (also 3.12),
so forcing a 3.10 *base* doesn't work. Instead, **Cell 1b creates a named conda env `lam` on Python 3.10**,
and every install/bake step runs inside it via `conda run -n lam`. No kernel restart; the env is deterministic.

> `files.upload()` / `files.download()` (Cells 5/7) run in the normal Colab kernel — they only move files on
> disk, so they don't need the env. Everything that imports LAM/torch/fbx goes through `conda run`.

### ⚠️ Honest status
Built from LAM's documented Linux install + source. The Python-3.10-base approach was tried on live Colab and
**failed** (condacolab gave 3.12); this `conda run` rewrite is the fix but is **itself not yet confirmed on a
full live run**. Biggest remaining risks: CUDA-compiled deps building inside the env (Cell 2), and the FBX
wheel importing in the env (Cell 4). **Zero-setup fallback:** LAM's
[ModelScope Space](https://www.modelscope.cn/studios/Damo_XR_Lab/LAM_Large_Avatar_Model) exports the OAC zip
server-side.


In [1]:
# Cell 1 — GPU + CUDA + Python report.  SUCCESS: T4 shown, 'CUDA available: True'.
!nvidia-smi
import sys, torch  # kernel torch (Colab's) — only used here to check the GPU; the env gets its own torch.
print('kernel python', '%d.%d.%d' % sys.version_info[:3], '| torch', torch.__version__,
      '| torch CUDA', torch.version.cuda, '| available', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then Restart and run all.'
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB')
print('Colab is Python 3.12; Cell 1b builds a Python-3.10 conda env for the FBX SDK. This is expected.')
# LAM-20K inference is light (~1.4s on A100) and fits T4 (~15GB). Host RAM (~12GB) is the tighter limit.


Tue Jun 23 08:48:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cell 1b — build the Python 3.10 conda env (`lam`)
Installs Miniforge to disk (no kernel restart) and creates env `lam` on Python 3.10. Defines `RUN` — the
`conda run` prefix used by every later install/bake cell. **Run this once before Cell 2.**

Why this works where condacolab didn't: we don't change the base/kernel Python at all. `conda create
python=3.10` resolves a real 3.10 interpreter, and the cp310 FBX wheel installs into it.


In [2]:
# Cell 1b — Python 3.10 env.  SUCCESS: prints 'env python: 3.10.x' and 'RUN = ...'.  (~3-5 min)
import os
CONDA = '/usr/local/miniforge3/bin/conda'
ENV = 'lam'
if not os.path.exists(CONDA):
    !wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/miniforge.sh
    !bash /tmp/miniforge.sh -b -p /usr/local/miniforge3
# create the 3.10 env if it isn't there yet
if not os.path.isdir(f'/usr/local/miniforge3/envs/{ENV}'):
    !{CONDA} create -y -n {ENV} python=3.10
# RUN = prefix for EVERY later package/install/bake command (streams output live)
RUN = f'{CONDA} run -n {ENV} --no-capture-output'
_v = !{CONDA} run -n {ENV} python -c "import sys;print('%d.%d.%d'%sys.version_info[:3])"
print('env python:', _v[-1] if _v else '??')
assert _v and _v[-1].startswith('3.10'), f'env is not 3.10: {_v}'
print('RUN =', RUN)


PREFIX=/usr/local/miniforge3
Unpacking bootstrapper...
Unpacking payload...
Extracting ca-certificates-2026.5.20-hbd8a1cb_0.conda
Extracting libgomp-15.2.0-he0feb66_19.conda
Extracting libzlib-1.3.2-h25fd6f3_2.conda
Extracting nlohmann_json-abi-3.12.0-h0f90c79_1.conda
Extracting pybind11-abi-11-hc364b38_1.conda
Extracting python_abi-3.13-8_cp313.conda
Extracting tzdata-2025c-hc9c84f9_1.conda
Extracting _openmp_mutex-4.5-20_gnu.conda
Extracting zstd-1.5.7-hb78ec9c_6.conda
Extracting ld_impl_linux-64-2.45.1-default_hbd61a6d_102.conda
Extracting libgcc-15.2.0-he0feb66_19.conda
Extracting bzip2-1.0.8-hda65f42_9.conda
Extracting c-ares-1.34.6-hb03c661_0.conda
Extracting keyutils-1.6.3-hb9d3cd8_0.conda
Extracting libexpat-2.8.1-hecca717_0.conda
Extracting libffi-3.5.2-h3435931_0.conda
Extracting libgcc-ng-15.2.0-h69a702a_19.conda
Extracting libiconv-1.18-h3b78370_2.conda
Extracting liblzma-5.8.3-hb03c661_0.conda
Extracting libmpdec-4.0.0-hb03c661_1.conda
Extracting libsqlite-3.53.1-h0c1763c_

## Cell 2 — install into the `lam` env with **prebuilt wheels** (no source-compile hang)
The earlier source builds hung ~45 min on `pytorch3d`. Runtime-dependency audit of the **bake/OAC path**
(`colab_bake_oac.py` → `app_lam` → `ModelLAM`/`gs_renderer` + `FlameTrackingSingleImage`):

| pkg | needed by bake? | how |
|-----|-----------------|-----|
| `pytorch3d` | **yes** (gs_renderer + flame model, imported at model build) | **prebuilt wheel** py310/cu121/pyt230 (0.7.6) |
| `diff_gaussian_rasterization` | **yes** (gs_renderer top-level import) | build (small single CUDA ext) w/ ninja + live `-v` + 15-min timeout |
| `nvdiffrast` | **yes** (vhap `export_as_nerf_dataset` top-level → flame tracking) | pip from git — **fast**, JIT-compiles its kernel at first use, not at install |
| `simple_knn` | **no** — imported nowhere in LAM | **dropped** |
| `pymcubes` | **no** — only `lam/runners/infer/lam.py`, not on the bake path | **dropped** |

So only **one** small thing still compiles (`diff_gaussian_rasterization`), guarded by a hard timeout +
live output so it can't silently hang. Everything else is wheels. **cu121 only** — the prebuilt pytorch3d
wheel is cu121 (Colab T4 is cu121); cu118 would need a source build and the cell will say so.

**Fail-loud:** every step is `subprocess.run(check=True[, timeout])`; non-zero **or** timeout raises and the
cell errors. `INSTALL DONE` only prints after an in-env `IMPORTS_OK` check imports the bake-critical packages
+ `ModelLAM` + `FlameTrackingSingleImage`.


In [3]:
# Cell 2 — install with prebuilt wheels (no source-compile hang).  FAILS LOUDLY (check=True + timeouts).
# SUCCESS: 'IMPORTS_OK' then 'INSTALL DONE'.  (~10-15 min)
import os, subprocess, torch
def sh(cmd, timeout=None):
    print('>>>', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, timeout=timeout)  # non-zero/timeout -> cell ERRORS
cu = (torch.version.cuda or '')
CU = 'cu121' if cu.startswith('12') else 'cu118'
WHL = f'https://download.pytorch.org/whl/{CU}'
print('runtime CUDA', cu, '->', CU)
if not os.path.isdir('/content/LAM'):
    sh('git clone https://github.com/aigc3d/LAM.git /content/LAM')
%cd /content/LAM
# 1) torch stack + xformers (wheels; LAM-pinned)
sh(f'{RUN} pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url {WHL}')
sh(f'{RUN} pip install -U xformers==0.0.26.post1 --index-url {WHL}')
# 2) build deps for the few small exts we still build (diff_gaussian_rasterization, FaceBoxes Cython)
sh(f'{RUN} pip install Cython ninja "numpy==1.23.0" setuptools wheel')
# 3) pytorch3d = PREBUILT wheel (kills the ~45-min source build). cu121 only.
assert CU == 'cu121', 'prebuilt pytorch3d wheel is cu121-only; on cu118 youd need a source build.'
sh(f'{RUN} pip install --no-index --no-cache-dir pytorch3d '
   f'-f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt230/download.html')
# 4) rest of requirements MINUS source-compile/unused pkgs (pytorch3d done; simple_knn+pymcubes dropped;
#    diff_gaussian_rasterization + nvdiffrast installed explicitly below).
reqs = [l.strip() for l in open('requirements.txt') if l.strip()]
drop = ('pytorch3d', 'simple-knn', 'simple_knn', 'diff-gaussian', 'diff_gaussian', 'nvdiffrast', 'pymcubes')
keep = [l for l in reqs if not any(d in l for d in drop)]
open('/tmp/req_min.txt', 'w').write('\n'.join(keep) + '\n')
print('dropped:', [l for l in reqs if l not in keep])
sh(f'{RUN} pip install --no-build-isolation -r /tmp/req_min.txt')
# 5) diff_gaussian_rasterization: small CUDA ext — ninja parallel, LIVE output (-v), hard 15-min timeout
os.environ['MAX_JOBS'] = '4'
sh(f'{RUN} pip install --no-build-isolation -v "git+https://github.com/ashawkey/diff-gaussian-rasterization"', timeout=900)
# 6) nvdiffrast: fast install (kernels JIT-compile at first render, not now)
sh(f'{RUN} pip install --no-build-isolation "nvdiffrast@git+https://github.com/ShenhanQian/nvdiffrast@backface-culling"', timeout=600)
# 7) FaceBoxesV2 Cython ext (fast; needs Cython, present)
sh(f'{RUN} sh -c "cd external/landmark_detection/FaceBoxesV2/utils && sh make.sh"', timeout=600)
# 8) IMPORTS_OK: only the packages the BAKE actually needs, plus the real entry points
open('/tmp/imp_check.py','w').write(
    'import torch, pytorch3d, diff_gaussian_rasterization, nvdiffrast.torch\n'
    'from lam.models import ModelLAM\n'
    'from tools.flame_tracking_single_image import FlameTrackingSingleImage\n'
    'print("IMPORTS_OK")\n')
sh(f'{RUN} python /tmp/imp_check.py', timeout=600)
print('INSTALL DONE')


runtime CUDA 12.8 -> cu121
>>> git clone https://github.com/aigc3d/LAM.git /content/LAM
/content/LAM
>>> /usr/local/miniforge3/bin/conda run -n lam --no-capture-output pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121
>>> /usr/local/miniforge3/bin/conda run -n lam --no-capture-output pip install -U xformers==0.0.26.post1 --index-url https://download.pytorch.org/whl/cu121
>>> /usr/local/miniforge3/bin/conda run -n lam --no-capture-output pip install Cython ninja "numpy==1.23.0" setuptools wheel
>>> /usr/local/miniforge3/bin/conda run -n lam --no-capture-output pip install --no-index --no-cache-dir pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt230/download.html


CalledProcessError: Command '/usr/local/miniforge3/bin/conda run -n lam --no-capture-output pip install --no-index --no-cache-dir pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt230/download.html' returned non-zero exit status 1.

In [4]:
!/usr/local/miniforge3/bin/conda run -n lam pip install pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt230/download.html

Looking in links: https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt230/download.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 143.2 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 32.5 MB/s  0:00:00
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-an

## Cell 3 — download weights + assets (HuggingFace)
`3DAIGC/LAM-20K` + `3DAIGC/LAM-assets` are **public** — token optional (helps rate limits). Paste yours in the
placeholder; **do not commit it**. (`huggingface-cli` runs in the env so it matches the installed hub version.)


In [5]:
# Cell 3 — weights + assets.  SUCCESS: 'WEIGHTS + ASSETS OK'.
# >>> OPTIONAL: paste your HF token (else anonymous). NEVER COMMIT THIS. <<<
HF_TOKEN = ''  # e.g. 'hf_xxx'
import os
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN  # conda run inherits this; huggingface-cli picks it up
# assets = sample_oac (template_file.fbx + animation.glb), sample_motion, sample_input + flame-tracking models
!{RUN} huggingface-cli download 3DAIGC/LAM-assets --local-dir ./tmp
!tar -xf ./tmp/LAM_assets.tar && rm ./tmp/LAM_assets.tar
!tar -xf ./tmp/thirdparty_models.tar && rm -r ./tmp/
!{RUN} huggingface-cli download 3DAIGC/LAM-20K --local-dir ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/
assert os.path.exists('model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors'), 'LAM-20K weights missing'
assert os.path.isdir('assets/sample_oac'), 'assets/sample_oac missing (needed for OAC: template_file.fbx + animation.glb)'
print('WEIGHTS + ASSETS OK')



Hint: A new version of huggingface_hub (1.20.1) is available! You are using version 1.19.0.
To update, run: hf update
Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help

ERROR conda.cli.main_run:execute(148): `conda run huggingface-cli download 3DAIGC/LAM-assets --local-dir ./tmp` failed. (See above for error)
tar: ./tmp/LAM_assets.tar: Cannot open: No such file or directory
tar: Error is not recoverable: exiting now
tar: ./tmp/thirdparty_models.tar: Cannot open: No such file or directory
tar: Error is not recoverable: exiting now

Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run py

AssertionError: LAM-20K weights missing

In [6]:
!/usr/local/miniforge3/bin/conda run -n lam bash -c "cd /content/LAM && hf download 3DAIGC/LAM-assets --local-dir ./tmp/ && tar -xf ./tmp/LAM_assets.tar && tar -xf ./tmp/thirdparty_models.tar && rm -r ./tmp/ && hf download 3DAIGC/LAM-20K --local-dir ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/"


✓ Downloaded
  path: /content/LAM/tmp
✓ Downloaded
  path: /content/LAM/model_zoo/lam_models/releases/lam/lam-20k/step_045500

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]Still waiting to acquire lock on /content/LAM/tmp/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)

Fetching 7 files: 100%|██████████| 7/7 [00:40<00:00,  5.76s/it]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Fetching 4 files: 100%|██████████| 4/4 [00:23<00:00,  5.81s/it]


## Cell 4 — Blender (headless) + FBX SDK (into the env)
`skin.glb` = ASCII FBX → *(FBX SDK)* binary FBX → *(Blender)* GLB; that Blender step also writes
`vertex_order.json`. Blender is a standalone binary (kernel-agnostic); the **cp310 FBX wheel installs into the
`lam` env** — which is the whole reason for Cell 1b.


In [7]:
# Cell 4 — Blender + FBX SDK.  SUCCESS: 'BLENDER OK' AND 'import fbx OK (in 3.10 env)'.
import os
BV = 'blender-4.0.2-linux-x64'  # guide-pinned; OAC needs Blender > 4.0
if not os.path.isdir(f'/content/{BV}'):
    !wget -q https://download.blender.org/release/Blender4.0/{BV}.tar.xz -O /content/blender.tar.xz
    !tar -xf /content/blender.tar.xz -C /content/
BLENDER = f'/content/{BV}/blender'
os.environ['BLENDER'] = BLENDER
r = os.system(f'{BLENDER} --background --version')
if r != 0:  # headless Blender still needs a few X libs present
    !apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libsm6 >/dev/null
    r = os.system(f'{BLENDER} --background --version')
assert r == 0, 'Blender headless failed — check the apt libs above.'
print('BLENDER OK:', BLENDER)
# FBX SDK (cp310) INTO the 3.10 env
!wget -q https://virutalbuy-public.oss-cn-hangzhou.aliyuncs.com/share/aigc3d/data/LAM/fbx-2020.3.4-cp310-cp310-manylinux1_x86_64.whl -O /content/fbx.whl
!{RUN} pip install -q /content/fbx.whl
_chk = !{CONDA} run -n {ENV} python -c "import fbx; print('FBX_OK')"
assert any('FBX_OK' in l for l in _chk), f'fbx import failed in env: {_chk}'
print('import fbx OK (in 3.10 env)')


BLENDER OK: /content/blender-4.0.2-linux-x64/blender
ERROR: Invalid wheel filename (wrong number of parts): 'fbx'
ERROR conda.cli.main_run:execute(148): `conda run pip install -q /content/fbx.whl` failed. (See above for error)
import fbx OK (in 3.10 env)


In [8]:
# Cell 5 — upload your fisherman portrait (runs in the Colab kernel — no env needed).
# SUCCESS: prints 'Saved: assets/sample_input/fisherman.jpg'. Front-facing, well-lit works best.
from google.colab import files
import os, shutil
up = files.upload()  # choose your image
src = list(up.keys())[0]
os.makedirs('assets/sample_input', exist_ok=True)
IMG = 'assets/sample_input/fisherman.jpg'
shutil.move(src, IMG)
print('Saved:', IMG)


Saving untitled_Google Imagen 3 Fast_2026-06-23_06-28-57.png to untitled_Google Imagen 3 Fast_2026-06-23_06-28-57.png
Saved: assets/sample_input/fisherman.jpg


## Cell 6 — bake + OAC export (headless, in the env)
Writes the gradio-free runner (adapted from `app_lam.py core_fn`) and runs it **via `conda run`** so it
executes on Python 3.10 with the FBX SDK + LAM deps. Output: `output/open_avatar_chat/<stem>.zip`
(inner folder == zip name). If it raises on a LAM internal, use the Gradio fallback below.


In [12]:
%%writefile colab_bake_oac.py
#!/usr/bin/env python3
"""
Headless LAM → OpenAvatarChat (OAC) bake — one image in, one OAC zip out.

This is the gradio-free path used by the Colab notebook (LAM_bake_oac_colab.ipynb).
It is adapted *faithfully* from `app_lam.py`'s `core_fn` OAC-export branch
(the part gated behind the "Export ZIP file for Chatting Avatar" checkbox), with
the video-rendering steps stripped out — we only need the four OAC files.

Run from inside the cloned LAM repo:
    python colab_bake_oac.py \
        --image assets/sample_input/fisherman.jpg \
        --blender_path /content/blender-4.0.2-linux-x64/blender \
        --motion auto

Output: ./output/open_avatar_chat/<image-stem>.zip containing
    <image-stem>/skin.glb, offset.ply, animation.glb, vertex_order.json
i.e. exactly LAM's OAC format (see ints-head-gs/docs/AVATAR_FORMAT.md). NOT
h5_render_data.zip.

⚠️ This orchestration mirrors LAM internals at a point in time. If LAM changes a
signature (infer_single_view / prepare_motion_seqs / save_shaped_mesh), this will
raise — fall back to the notebook's Gradio cell, which runs LAM's own code.
"""
import argparse
import os
import sys
import shutil
import zipfile
from glob import glob
from pathlib import Path


def find_motion(motion_arg: str) -> str:
    """Pick a driving motion-sequence dir (provides flame shape + render params)."""
    if motion_arg and motion_arg != "auto":
        d = f"./assets/sample_motion/export/{motion_arg}"
        assert os.path.isdir(d), f"motion not found: {d}"
        return d
    cands = sorted(glob("./assets/sample_motion/export/*/"))
    assert cands, "no sample motions under assets/sample_motion/export/ — did the assets download succeed?"
    # prefer the one LAM's own inference.sh uses, if present
    for c in cands:
        if "Look_In_My_Eyes" in c:
            return c.rstrip("/")
    return cands[0].rstrip("/")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--image", required=True, help="input portrait, e.g. assets/sample_input/fisherman.jpg")
    ap.add_argument("--blender_path", required=True, help="path to Blender >4.0 executable")
    ap.add_argument("--motion", default="auto", help="motion seq name under assets/sample_motion/export, or 'auto'")
    args = ap.parse_args()

    assert os.path.exists(args.image), f"image not found: {args.image}"
    assert os.path.exists(args.blender_path), f"blender not found: {args.blender_path}"

    # --- env + config, copied from app_lam.launch_gradio_app -----------------
    os.environ.update({
        "APP_ENABLED": "1",
        "APP_MODEL_NAME": "./model_zoo/lam_models/releases/lam/lam-20k/step_045500/",
        "APP_INFER": "./configs/inference/lam-20k-8gpu.yaml",
        "APP_TYPE": "infer.lam",
        "NUMBA_THREADING_LAYER": "omp",
    })

    import torch
    import app_lam  # importing does NOT launch gradio (that's under __main__)
    from tools.generateARKITGLBWithBlender import generate_glb
    from lam.runners.infer.head_utils import prepare_motion_seqs, preprocess_image

    # parse_configs reads --blender_path off sys.argv; hand it a clean argv.
    sys.argv = ["colab_bake_oac.py", "--blender_path", args.blender_path]
    cfg, _ = app_lam.parse_configs()

    print("building model + flame tracking…")
    lam = app_lam._build_model(cfg)
    lam.to("cuda").eval()

    from tools.flame_tracking_single_image import FlameTrackingSingleImage
    flametracking = FlameTrackingSingleImage(
        output_dir="output/tracking",
        alignment_model_path="./model_zoo/flame_tracking_models/68_keypoints_model.pkl",
        vgghead_model_path="./model_zoo/flame_tracking_models/vgghead/vgg_heads_l.trcd",
        human_matting_path="./model_zoo/flame_tracking_models/matting/stylematte_synth.pt",
        facebox_model_path="./model_zoo/flame_tracking_models/FaceBoxesV2.pth",
        detect_iris_landmarks=False,
    )

    motion_seqs_dir = find_motion(args.motion)
    base_iid = os.path.basename(args.image).split(".")[0]
    print(f"image={args.image}  iid={base_iid}  motion={motion_seqs_dir}")

    # --- flame tracking on the input image (core_fn steps) -------------------
    tmp_dir = "output/_bake_tmp"
    os.makedirs(tmp_dir, exist_ok=True)
    image_raw = os.path.join(tmp_dir, "raw.png")
    from PIL import Image
    with Image.open(args.image).convert("RGB") as im:
        im.save(image_raw)

    assert flametracking.preprocess(image_raw) == 0, "flametracking preprocess failed"
    assert flametracking.optimize() == 0, "flametracking optimize failed"
    rc, output_dir = flametracking.export()
    assert rc == 0, "flametracking export failed"

    image_path = os.path.join(output_dir, "images/00000_00.png")
    mask_path = os.path.join(output_dir, "fg_masks/00000_00.png")

    aspect_standard = 1.0 / 1.0
    image, _, _, shape_param = preprocess_image(
        image_path, mask_path=mask_path, intr=None, pad_ratio=0, bg_color=1.0,
        max_tgt_size=None, aspect_standard=aspect_standard, enlarge_ratio=[1.0, 1.0],
        render_tgt_size=cfg.source_size, multiply=14, need_mask=True, get_shape_param=True,
    )

    src = image_path.split("/")[-3]
    driven = motion_seqs_dir.split("/")[-2]
    motion_seq = prepare_motion_seqs(
        motion_seqs_dir, None, save_root=tmp_dir, fps=30, bg_color=1.0,
        aspect_standard=aspect_standard, enlarge_ratio=[1.0, 1, 0],
        render_image_res=cfg.render_size, multiply=16, need_mask=False,
        vis_motion=False, shape_param=shape_param, test_sample=False,
        cross_id=False, src_driven=[src, driven],
    )

    # --- inference → canonical gaussians -------------------------------------
    device, dtype = "cuda", torch.float32
    motion_seq["flame_params"]["betas"] = shape_param.unsqueeze(0)
    print("running LAM inference…")
    with torch.no_grad():
        res = lam.infer_single_view(
            image.unsqueeze(0).to(device, dtype), None, None,
            render_c2ws=motion_seq["render_c2ws"].to(device),
            render_intrs=motion_seq["render_intrs"].to(device),
            render_bg_colors=motion_seq["render_bg_colors"].to(device),
            flame_params={k: v.to(device) for k, v in motion_seq["flame_params"].items()},
        )

    # --- OAC export (app_lam.py lines ~304-342, minus the video) -------------
    oac_dir = os.path.join("./output/open_avatar_chat", base_iid)
    os.makedirs(oac_dir, exist_ok=True)
    print("writing offset.ply…")
    saved_head_path = lam.renderer.flame_model.save_shaped_mesh(
        shape_param.unsqueeze(0).cuda(), fd=oac_dir,
    )
    res["cano_gs_lst"][0].save_ply(os.path.join(oac_dir, "offset.ply"), rgb2sh=False, offset2xyz=True)

    print("generating skin.glb via Blender + FBX SDK (also writes vertex_order.json)…")
    generate_glb(
        input_mesh=Path(saved_head_path),
        template_fbx=Path("./assets/sample_oac/template_file.fbx"),
        output_glb=Path(os.path.join(oac_dir, "skin.glb")),
        blender_exec=Path(cfg.blender_path),
    )
    shutil.copy("./assets/sample_oac/animation.glb", os.path.join(oac_dir, "animation.glb"))
    if os.path.exists(saved_head_path):
        os.remove(saved_head_path)

    # --- validate the 4 OAC files, then zip with inner-folder == zip-name ----
    required = ["skin.glb", "offset.ply", "animation.glb", "vertex_order.json"]
    missing = [f for f in required if not os.path.exists(os.path.join(oac_dir, f))]
    assert not missing, f"OAC export incomplete, missing: {missing} (vertex_order.json comes from generate_glb step 4)"

    out_zip = os.path.join("./output/open_avatar_chat", base_iid + ".zip")
    if os.path.exists(out_zip):
        os.remove(out_zip)
    with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for f in required:
            z.write(os.path.join(oac_dir, f), arcname=os.path.join(base_iid, f))

    print("\n✅ OAC zip ready:", os.path.abspath(out_zip))
    print("   inner folder:", base_iid, "(== zip name; matches docs/AVATAR_FORMAT.md)")
    print("   contents:", ", ".join(required))


if __name__ == "__main__":
    main()


Overwriting colab_bake_oac.py


In [14]:
!/usr/local/miniforge3/bin/conda run -n lam pip install opencv-python-headless

  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 70.6 MB/s  0:00:00
Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.0
    Uninstalling numpy-1.23.0:
      Successfully uninstalled numpy-1.23.0



In [16]:
!/usr/local/miniforge3/bin/conda run -n lam pip install gradio "numpy==1.23.0"

  Using cached numpy-1.23.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.2 kB)
Using cached numpy-1.23.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 72.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 33.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 123.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 139.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 88.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 63.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.3 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour i

In [18]:
!/usr/local/miniforge3/bin/conda run -n lam pip install omegaconf einops roma "numpy==1.23.0"

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144590 sha256=a4b40540e0797c3b970f8bfe2028d200799b461e305b3adf5fdbd869c7cb85a3
  Stored in directory: /root/.cache/pip/wheels/12/93/dd/1f6a127edc45659556564c5730f6d4e300888f4bca2d4c5a88
Successfully built antlr4-python3-runtime



In [20]:
!/usr/local/miniforge3/bin/conda run -n lam pip install moviepy imageio imageio-ffmpeg "numpy==1.23.0"

INFO: pip is looking at multiple versions of moviepy to determine which version is compatible with other requirements. This could take a while.
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.5/29.5 MB 176.3 MB/s  0:00:00
  Created wheel for moviepy: filename=moviepy-1.0.3-py3-none-any.whl size=110796 sha256=fd6f3204b337ba3f032181c22edde86e08bc8bb4980014528cc4aaee470b25c5
  Stored in directory: /root/.cache/pip/wheels/96/32/2d/e10123bd88fbfc02fed53cc18c80a171d3c87479ed845fa7c1
Successfully built moviepy



In [22]:
!/usr/local/miniforge3/bin/conda run -n lam pip install tyro chumpy lpips face-alignment "numpy==1.23.0"

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
ERROR: Failed to build 'chumpy' when getting requirements to build wheel
ERROR conda.cli.main_run:execute(148): `conda run pip install tyro chumpy lpips face-alignment numpy==1.23.0` failed. (See above for error)


In [24]:
!/usr/local/miniforge3/bin/conda run -n lam pip install tyro lpips face-alignment "numpy==1.23.0"

  Using cached tyro-1.0.15-py3-none-any.whl.metadata (12 kB)
INFO: pip is looking at multiple versions of scipy to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of scikit-image to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 115.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 145.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 69.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 73.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 168.2 MB/s  0:00:00



In [25]:
!/usr/local/miniforge3/bin/conda run -n lam pip install --no-build-isolation chumpy "numpy==1.23.0"


  Using cached chumpy-0.70.tar.gz (50 kB)
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for chumpy: filename=chumpy-0.70-py3-none-any.whl size=58302 sha256=0e36e1bbe3fecc4f0b5e085d797fc11b72a4ee3a36de6f85e811ed8fb8d34e30
  Stored in directory: /root/.cache/pip/wheels/e0/c1/ef/29ba7be03653a29ef6f2c3e1956d6c4d8877f2b243af411db1
Successfully built chumpy


In [27]:
!/usr/local/miniforge3/bin/conda run -n lam pip install loguru scikit-image kornia "numpy==1.23.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 38.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 94.2 MB/s  0:00:00



In [29]:
!/usr/local/miniforge3/bin/conda run -n lam pip install transformers accelerate safetensors "numpy==1.23.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 102.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 107.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.1/794.1 kB 38.5 MB/s  0:00:00



In [31]:
!/usr/local/miniforge3/bin/conda run -n lam pip install "transformers==4.40.0" "numpy==1.23.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 152.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 141.6 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.20.1
    Uninstalling huggingface_hub-1.20.1:
      Successfully uninstalled huggingface_hub-1.20.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.1
    Uninstalling transformers-5.12.1:
      Successfully uninstalled transformers-5.12.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but

In [32]:
!/usr/local/miniforge3/bin/conda run -n lam bash -c "cd /content/LAM/external/landmark_detection/FaceBoxesV2/utils && python build.py build_ext --inplace"

Compiling nms/cpu_nms.pyx because it changed.
[1/1] Cythonizing nms/cpu_nms.pyx
nms/cpu_nms.c: In function ‘__pyx_pf_3nms_7cpu_nms_2cpu_soft_nms’:
nms/cpu_nms.c:5976:32: warning: comparison of integer expressions of different signedness: ‘int’ and ‘unsigned int’ [-Wsign-compare]
 5976 |       __pyx_t_9 = (__pyx_v_pos < __pyx_v_N);
      |                                ^
nms/cpu_nms.c:6487:32: warning: comparison of integer expressions of different signedness: ‘int’ and ‘unsigned int’ [-Wsign-compare]
 6487 |       __pyx_t_9 = (__pyx_v_pos < __pyx_v_N);
      |                                ^


In [34]:
!/usr/local/miniforge3/bin/conda run -n lam pip install matplotlib trimesh "numpy==1.23.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 149.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 38.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 170.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 91.5 MB/s  0:00:00



In [36]:
!MPLBACKEND=Agg /usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion auto"

Traceback (most recent call last):
  File "/content/LAM/colab_bake_oac.py", line 179, in <module>
    main()
  File "/content/LAM/colab_bake_oac.py", line 69, in main
    import app_lam  # importing does NOT launch gradio (that's under __main__)
  File "/content/LAM/app_lam.py", line 32, in <module>
    from tools.flame_tracking_single_image import FlameTrackingSingleImage
  File "/content/LAM/tools/flame_tracking_single_image.py", line 22, in <module>
    from vhap.export_as_nerf_dataset import (NeRFDatasetWriter,
  File "/content/LAM/vhap/export_as_nerf_dataset.py", line 32, in <module>
    from vhap.util.render_nvdiffrast import NVDiffRenderer
  File "/content/LAM/vhap/util/render_nvdiffrast.py", line 12, in <module>
    import nvdiffrast.torch as dr
ModuleNotFoundError: No module named 'nvdiffrast'
ERROR conda.cli.main_run:execute(148): `conda run bash -c cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path /content/ble

In [37]:
!/usr/local/miniforge3/bin/conda run -n lam pip install ninja git+https://github.com/NVlabs/nvdiffrast.git "numpy==1.23.0"

  Cloning https://github.com/NVlabs/nvdiffrast.git to /tmp/pip-req-build-oudp0npa
  Resolved https://github.com/NVlabs/nvdiffrast.git to commit 253ac4fcea7de5f396371124af597e6cc957bfae
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
  Running command git clone --filter=blob:none --quiet https://github.com/NVlabs/nvdiffrast.git /tmp/pip-req-build-oudp0npa
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
ERROR: Failed to build 'git+https://github.com/NVlabs/nvdiffrast.git' when getting requirements to build wheel
ERROR conda.cli.main_run:execute(148): `conda run pip install ninja git+https://github.com/NVlabs/nvdiffrast.git numpy==1.

In [38]:
!/usr/local/miniforge3/bin/conda run -n lam pip install --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git "numpy==1.23.0"

  Cloning https://github.com/NVlabs/nvdiffrast.git to /tmp/pip-req-build-8w2g72ck
  Resolved https://github.com/NVlabs/nvdiffrast.git to commit 253ac4fcea7de5f396371124af597e6cc957bfae
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for nvdiffrast: filename=nvdiffrast-0.4.0-cp310-cp310-linux_x86_64.whl size=1035283 sha256=425ee8b73695a5b7af62262bf5ad8d95d983b09d5cceb50b8189c150c4ae10f6
  Stored in directory: /tmp/pip-ephem-wheel-cache-71xm5_ir/wheels/24/2b/98/f611ce0d4062793b78daf724e6b47ee800c9a2d3e1ff4b06fa
Successfully built nvdiffrast
  Running command git clone --filter=blob:none --quiet https://github.com/NVlabs/nvdiffrast.git /tmp/pip-req-build-8w2g72ck


In [39]:
!/usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion auto"

Traceback (most recent call last):
  File "/content/LAM/colab_bake_oac.py", line 179, in <module>
    main()
  File "/content/LAM/colab_bake_oac.py", line 69, in main
    import app_lam  # importing does NOT launch gradio (that's under __main__)
  File "/content/LAM/app_lam.py", line 32, in <module>
    from tools.flame_tracking_single_image import FlameTrackingSingleImage
  File "/content/LAM/tools/flame_tracking_single_image.py", line 24, in <module>
    from vhap.model.tracker import GlobalTracker
  File "/content/LAM/vhap/model/tracker.py", line 21, in <module>
    from torch.utils.tensorboard import SummaryWriter
  File "/usr/local/miniforge3/envs/lam/lib/python3.10/site-packages/torch/utils/tensorboard/__init__.py", line 1, in <module>
    import tensorboard
ModuleNotFoundError: No module named 'tensorboard'
ERROR conda.cli.main_run:execute(148): `conda run bash -c cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path 

In [40]:
!/usr/local/miniforge3/bin/conda run -n lam pip install tensorboard "numpy==1.23.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 128.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 125.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 135.7 MB/s  0:00:00



In [41]:
!/usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion auto"

Traceback (most recent call last):
  File "/content/LAM/colab_bake_oac.py", line 179, in <module>
    main()
  File "/content/LAM/colab_bake_oac.py", line 69, in main
    import app_lam  # importing does NOT launch gradio (that's under __main__)
  File "/content/LAM/app_lam.py", line 34, in <module>
    from lam.runners.infer.head_utils import prepare_motion_seqs, preprocess_image
  File "/content/LAM/lam/runners/__init__.py", line 20, in <module>
    from .infer import *
  File "/content/LAM/lam/runners/infer/__init__.py", line 15, in <module>
    from .lam import LAMInferrer
  File "/content/LAM/lam/runners/infer/lam.py", line 19, in <module>
    import mcubes
ModuleNotFoundError: No module named 'mcubes'
ERROR conda.cli.main_run:execute(148): `conda run bash -c cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion auto` failed. (See above for error)


In [42]:
!/usr/local/miniforge3/bin/conda run -n lam pip install --no-build-isolation pymcubes "numpy==1.23.0"


In [43]:
!/usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion auto"

Traceback (most recent call last):
  File "/content/LAM/colab_bake_oac.py", line 179, in <module>
    main()
  File "/content/LAM/colab_bake_oac.py", line 69, in main
    import app_lam  # importing does NOT launch gradio (that's under __main__)
  File "/content/LAM/app_lam.py", line 34, in <module>
    from lam.runners.infer.head_utils import prepare_motion_seqs, preprocess_image
  File "/content/LAM/lam/runners/__init__.py", line 20, in <module>
    from .infer import *
  File "/content/LAM/lam/runners/infer/__init__.py", line 15, in <module>
    from .lam import LAMInferrer
  File "/content/LAM/lam/runners/infer/lam.py", line 37, in <module>
    from lam.models.modeling_lam import ModelLAM
  File "/content/LAM/lam/models/__init__.py", line 16, in <module>
    from .modeling_lam import ModelLAM
  File "/content/LAM/lam/models/modeling_lam.py", line 25, in <module>
    from .transformer import TransformerDecoder
  File "/content/LAM/lam/models/transformer.py", line 21, in <module>
   

In [44]:
!/usr/local/miniforge3/bin/conda run -n lam pip install "diffusers==0.27.2" "numpy==1.23.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 89.5 MB/s  0:00:00



In [45]:
!/usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion auto"

Traceback (most recent call last):
  File "/content/LAM/colab_bake_oac.py", line 179, in <module>
    main()
  File "/content/LAM/colab_bake_oac.py", line 69, in main
    import app_lam  # importing does NOT launch gradio (that's under __main__)
  File "/content/LAM/app_lam.py", line 34, in <module>
    from lam.runners.infer.head_utils import prepare_motion_seqs, preprocess_image
  File "/content/LAM/lam/runners/__init__.py", line 20, in <module>
    from .infer import *
  File "/content/LAM/lam/runners/infer/__init__.py", line 15, in <module>
    from .lam import LAMInferrer
  File "/content/LAM/lam/runners/infer/lam.py", line 37, in <module>
    from lam.models.modeling_lam import ModelLAM
  File "/content/LAM/lam/models/__init__.py", line 16, in <module>
    from .modeling_lam import ModelLAM
  File "/content/LAM/lam/models/modeling_lam.py", line 25, in <module>
    from .transformer import TransformerDecoder
  File "/content/LAM/lam/models/transformer.py", line 21, in <module>
   

In [46]:
!/usr/local/miniforge3/bin/conda run -n lam pip install "diffusers==0.31.0" "numpy==1.23.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 90.4 MB/s  0:00:00
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.27.2
    Uninstalling diffusers-0.27.2:
      Successfully uninstalled diffusers-0.27.2


In [47]:
!/usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion auto"

Traceback (most recent call last):
  File "/content/LAM/lam/models/rendering/gs_renderer.py", line 20, in <module>
    from diff_gaussian_rasterization_wda import GaussianRasterizationSettings, GaussianRasterizer
ModuleNotFoundError: No module named 'diff_gaussian_rasterization_wda'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/LAM/colab_bake_oac.py", line 179, in <module>
    main()
  File "/content/LAM/colab_bake_oac.py", line 69, in main
    import app_lam  # importing does NOT launch gradio (that's under __main__)
  File "/content/LAM/app_lam.py", line 34, in <module>
    from lam.runners.infer.head_utils import prepare_motion_seqs, preprocess_image
  File "/content/LAM/lam/runners/__init__.py", line 20, in <module>
    from .infer import *
  File "/content/LAM/lam/runners/infer/__init__.py", line 15, in <module>
    from .lam import LAMInferrer
  File "/content/LAM/lam/runners/infer/lam.py", line 37, in <m

In [48]:
!/usr/local/miniforge3/bin/conda run -n lam pip install --no-build-isolation git+https://github.com/graphdeco-inria/diff-gaussian-rasterization.git "numpy==1.23.0"

  Cloning https://github.com/graphdeco-inria/diff-gaussian-rasterization.git to /tmp/pip-req-build-d3hc0iom
  Resolved https://github.com/graphdeco-inria/diff-gaussian-rasterization.git to commit 59f5f77e3ddbac3ed9db93ec2cfe99ed6c5d121d
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for diff_gaussian_rasterization: filename=diff_gaussian_rasterization-0.0.0-cp310-cp310-linux_x86_64.whl size=696222 sha256=c3cc610a4925cb741532a6742a575a71dc84df9f362590a7b55ab908c17756c6
  Stored in directory: /tmp/pip-ephem-wheel-cache-1sxdojfn/wheels/d9/26/6d/b38c8286b71d9d23fbe23d44250de8de64733626425c954c94
Successfully built diff_gaussian_rasterization
  Running command git clone --filter=blob:none --quiet https://github.com/graphdeco-inria/diff-gaussian-rasterization.git /tmp/pip-req-build-d3hc0iom
  Running command git submodule update --init --recursive -q


In [49]:
!/usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion auto"

Traceback (most recent call last):
  File "/content/LAM/colab_bake_oac.py", line 179, in <module>
    main()
  File "/content/LAM/colab_bake_oac.py", line 69, in main
    import app_lam  # importing does NOT launch gradio (that's under __main__)
  File "/content/LAM/app_lam.py", line 34, in <module>
    from lam.runners.infer.head_utils import prepare_motion_seqs, preprocess_image
  File "/content/LAM/lam/runners/__init__.py", line 20, in <module>
    from .infer import *
  File "/content/LAM/lam/runners/infer/__init__.py", line 15, in <module>
    from .lam import LAMInferrer
  File "/content/LAM/lam/runners/infer/lam.py", line 37, in <module>
    from lam.models.modeling_lam import ModelLAM
  File "/content/LAM/lam/models/__init__.py", line 16, in <module>
    from .modeling_lam import ModelLAM
  File "/content/LAM/lam/models/modeling_lam.py", line 26, in <module>
    from lam.models.rendering.gs_renderer import GS3DRenderer, PointEmbed
  File "/content/LAM/lam/models/rendering/gs_re

In [50]:
!/usr/local/miniforge3/bin/conda run -n lam pip install plyfile "numpy==1.23.0"

INFO: pip is looking at multiple versions of plyfile to determine which version is compatible with other requirements. This could take a while.


In [51]:
!/usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion auto"

Traceback (most recent call last):
  File "/content/LAM/colab_bake_oac.py", line 179, in <module>
    main()
  File "/content/LAM/colab_bake_oac.py", line 69, in main
    import app_lam  # importing does NOT launch gradio (that's under __main__)
  File "/content/LAM/app_lam.py", line 34, in <module>
    from lam.runners.infer.head_utils import prepare_motion_seqs, preprocess_image
  File "/content/LAM/lam/runners/__init__.py", line 20, in <module>
    from .infer import *
  File "/content/LAM/lam/runners/infer/__init__.py", line 15, in <module>
    from .lam import LAMInferrer
  File "/content/LAM/lam/runners/infer/lam.py", line 37, in <module>
    from lam.models.modeling_lam import ModelLAM
  File "/content/LAM/lam/models/__init__.py", line 16, in <module>
    from .modeling_lam import ModelLAM
  File "/content/LAM/lam/models/modeling_lam.py", line 26, in <module>
    from lam.models.rendering.gs_renderer import GS3DRenderer, PointEmbed
  File "/content/LAM/lam/models/rendering/gs_re

In [52]:
!/usr/local/miniforge3/bin/conda run -n lam pip install jaxtyping "numpy==1.23.0"

In [53]:
!/usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion auto"

Traceback (most recent call last):
  File "/content/LAM/tools/generateARKITGLBWithBlender.py", line 21, in <module>
    import fbx
ModuleNotFoundError: No module named 'fbx'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/LAM/colab_bake_oac.py", line 179, in <module>
    main()
  File "/content/LAM/colab_bake_oac.py", line 70, in main
    from tools.generateARKITGLBWithBlender import generate_glb
  File "/content/LAM/tools/generateARKITGLBWithBlender.py", line 23, in <module>
    raise RuntimeError(
RuntimeError: FBX SDK required: https://www.autodesk.com/developer-network/platform-technologies/fbx-sdk-2020-2
ERROR conda.cli.main_run:execute(148): `conda run bash -c cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion auto` failed. (See above for error)


In [54]:
!/usr/local/miniforge3/bin/conda run -n lam python -c "import fbx; print('FBX OK', fbx.__file__)"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'fbx'
ERROR conda.cli.main_run:execute(148): `conda run python -c import fbx; print('FBX OK', fbx.__file__)` failed. (See above for error)


In [55]:
!cat /content/LAM/tools/AVATAR_EXPORT_GUIDE.md 2>/dev/null | head -40; ls -la /content/*.whl /content/fbx* 2>/dev/null

## Export Chatting Avatar Guide
### 🛠️ Environment Setup
#### Prerequisites
```
Python FBX SDK 2020.2+
Blender (version > 4.0.0)
```
#### Step1: download and install python fbx-sdk and other requirements
```bash
# FBX SDK: https://www.autodesk.com/developer-network/platform-technologies/fbx-sdk-2020-2
# Download FBX SDK installation package, example for Linux
wget https://virutalbuy-public.oss-cn-hangzhou.aliyuncs.com/share/aigc3d/data/LAM/fbxsdk_linux.tar
tar -xf fbxsdk_linux.tar
sh tools/install_fbx_sdk.sh

# Or download and install the FBX-SDK Wheel built by us
wget https://virutalbuy-public.oss-cn-hangzhou.aliyuncs.com/share/aigc3d/data/LAM/fbx-2020.3.4-cp310-cp310-manylinux1_x86_64.whl
pip install fbx-2020.3.4-cp310-cp310-manylinux1_x86_64.whl

# Install other requirements
pip install pathlib
pip install patool
```
#### Step2: download blender
```bash
# Download latest Blender (>=4.0.0)
# Choose appropriate version from: https://www.blender.org/download/
# Example for Linux
wget h

In [56]:
!/usr/local/miniforge3/bin/conda run -n lam pip install /content/fbx.whl && /usr/local/miniforge3/bin/conda run -n lam pip install pathlib patool && echo "--- verify ---" && /usr/local/miniforge3/bin/conda run -n lam python -c "import fbx; print('FBX OK')"

ERROR: Invalid wheel filename (wrong number of parts): 'fbx'
ERROR conda.cli.main_run:execute(148): `conda run pip install /content/fbx.whl` failed. (See above for error)


In [57]:
!cp /content/fbx.whl /content/fbx-2020.3.4-cp310-cp310-manylinux1_x86_64.whl && /usr/local/miniforge3/bin/conda run -n lam pip install /content/fbx-2020.3.4-cp310-cp310-manylinux1_x86_64.whl && /usr/local/miniforge3/bin/conda run -n lam pip install pathlib patool && echo "--- verify ---" && /usr/local/miniforge3/bin/conda run -n lam python -c "import fbx; print('FBX OK')"

Processing /content/fbx-2020.3.4-cp310-cp310-manylinux1_x86_64.whl

--- verify ---
FBX OK


In [58]:
!/usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion auto"

building model + flame tracking…
  warnings.warn("xFormers is available (SwiGLU)")

  warnings.warn("xFormers is available (Attention)")

  warnings.warn("xFormers is available (Block)")

INFO: using MLP layer as FFN
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_reg4_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitl14_reg4_pretrain.pth

skip_decoder: True 
#########scale sphere:False, add_teeth:False
 Render rgb: True 
face_upsampled:(39904, 3), face_ori:torch.Size([9976, 3]),                 vertex_num_upsampled:20018, vertex_num_ori:5023
loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
finish loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
2026-06-23 09:47:37.376 | INFO     | tools.flame_tracking_single_image:__init__:69 - Output Directory: output/tracking
2026-06-23 09:47:37.376 | INFO     | tools.flame_tracking_single_ima

In [59]:
!find /content/LAM/assets -name "transforms.json" 2>/dev/null; echo "--- sample_motion dir ---"; ls -R /content/LAM/assets/sample_motion 2>/dev/null | head -40

/content/LAM/assets/sample_motion/export/D_ANgelo_Dinero/transforms.json
/content/LAM/assets/sample_motion/export/Joe_Biden/transforms.json
/content/LAM/assets/sample_motion/export/The_Shawshank_Redemption/transforms.json
/content/LAM/assets/sample_motion/export/Speeding_Scandal/transforms.json
/content/LAM/assets/sample_motion/export/Anti_Drugs/transforms.json
/content/LAM/assets/sample_motion/export/I_Am_Iron_Man/transforms.json
/content/LAM/assets/sample_motion/export/Pen_Pineapple_Apple_Pen/transforms.json
/content/LAM/assets/sample_motion/export/Michael_Wayne_Rosen/transforms.json
/content/LAM/assets/sample_motion/export/Look_In_My_Eyes/transforms.json
/content/LAM/assets/sample_motion/export/Donald_Trump/transforms.json
/content/LAM/assets/sample_motion/export/Taylor_Swift/transforms.json
/content/LAM/assets/sample_motion/export/GEM/transforms.json
--- sample_motion dir ---
/content/LAM/assets/sample_motion:
export
export_expression_json.py

/content/LAM/assets/sample_motion/expo

In [60]:
!/usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion assets/sample_motion/export/Look_In_My_Eyes"

building model + flame tracking…
  warnings.warn("xFormers is available (SwiGLU)")

  warnings.warn("xFormers is available (Attention)")

  warnings.warn("xFormers is available (Block)")

INFO: using MLP layer as FFN
skip_decoder: True 
#########scale sphere:False, add_teeth:False
 Render rgb: True 
face_upsampled:(39904, 3), face_ori:torch.Size([9976, 3]),                 vertex_num_upsampled:20018, vertex_num_ori:5023
loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
finish loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
2026-06-23 09:50:11.793 | INFO     | tools.flame_tracking_single_image:__init__:69 - Output Directory: output/tracking
2026-06-23 09:50:11.794 | INFO     | tools.flame_tracking_single_image:__init__:72 - Loading Pre-trained Models...
  warnings.warn("'torch.load' received a zip file that looks like a TorchScript archive"

2026-06-23 09:50:19.095 | INFO   

In [61]:
!/usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion Look_In_My_Eyes"

building model + flame tracking…
  warnings.warn("xFormers is available (SwiGLU)")

  warnings.warn("xFormers is available (Attention)")

  warnings.warn("xFormers is available (Block)")

INFO: using MLP layer as FFN
skip_decoder: True 
#########scale sphere:False, add_teeth:False
 Render rgb: True 
face_upsampled:(39904, 3), face_ori:torch.Size([9976, 3]),                 vertex_num_upsampled:20018, vertex_num_ori:5023
loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
finish loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
2026-06-23 09:51:19.075 | INFO     | tools.flame_tracking_single_image:__init__:69 - Output Directory: output/tracking
2026-06-23 09:51:19.075 | INFO     | tools.flame_tracking_single_image:__init__:72 - Loading Pre-trained Models...
  warnings.warn("'torch.load' received a zip file that looks like a TorchScript archive"

2026-06-23 09:51:27.111 | INFO   

In [62]:
!cd /content/LAM && wget -q -O colab_bake_oac.py https://raw.githubusercontent.com/IntsI/ints-head-gs/main/tools/colab_bake_oac.py && echo "--- fetched, running bake ---" && /usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion Look_In_My_Eyes"

--- fetched, running bake ---
building model + flame tracking…
  warnings.warn("xFormers is available (SwiGLU)")

  warnings.warn("xFormers is available (Attention)")

  warnings.warn("xFormers is available (Block)")

INFO: using MLP layer as FFN
skip_decoder: True 
#########scale sphere:False, add_teeth:False
 Render rgb: True 
face_upsampled:(39904, 3), face_ori:torch.Size([9976, 3]),                 vertex_num_upsampled:20018, vertex_num_ori:5023
loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
finish loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
2026-06-23 09:58:04.818 | INFO     | tools.flame_tracking_single_image:__init__:69 - Output Directory: output/tracking
2026-06-23 09:58:04.818 | INFO     | tools.flame_tracking_single_image:__init__:72 - Loading Pre-trained Models...
  warnings.warn("'torch.load' received a zip file that looks like a TorchScript archive"

202

In [63]:
!grep -ri "diff.gaussian\|rasterization\|diff_gaussian" /content/LAM/requirements.txt /content/LAM/lam/models/rendering/gs_renderer.py | head -20

/content/LAM/requirements.txt:git+https://github.com/ashawkey/diff-gaussian-rasterization/
/content/LAM/lam/models/rendering/gs_renderer.py:    from diff_gaussian_rasterization_wda import GaussianRasterizationSettings, GaussianRasterizer
/content/LAM/lam/models/rendering/gs_renderer.py:    from diff_gaussian_rasterization import GaussianRasterizationSettings, GaussianRasterizer
/content/LAM/lam/models/rendering/gs_renderer.py:    RasterizationSettings,
/content/LAM/lam/models/rendering/gs_renderer.py:        # Set up rasterization configuration
/content/LAM/lam/models/rendering/gs_renderer.py:        GSRSettings = GaussianRasterizationSettings


In [64]:
!/usr/local/miniforge3/bin/conda run -n lam pip uninstall -y diff_gaussian_rasterization diff-gaussian-rasterization 2>/dev/null; /usr/local/miniforge3/bin/conda run -n lam pip install --no-build-isolation git+https://github.com/ashawkey/diff-gaussian-rasterization.git "numpy==1.23.0"

Found existing installation: diff_gaussian_rasterization 0.0.0
Uninstalling diff_gaussian_rasterization-0.0.0:
  Successfully uninstalled diff_gaussian_rasterization-0.0.0
  Cloning https://github.com/ashawkey/diff-gaussian-rasterization.git to /tmp/pip-req-build-jvb9mze_
  Resolved https://github.com/ashawkey/diff-gaussian-rasterization.git to commit 8829d14f814fccdaf840b7b0f3021a616583c0a1
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for diff_gaussian_rasterization: filename=diff_gaussian_rasterization-0.0.0-cp310-cp310-linux_x86_64.whl size=697144 sha256=8123f536052b7c59e27ab4fce1e77b82bb4802c13ed7529af702dc13ecc18645
  Stored in directory: /tmp/pip-ephem-wheel-cache-59a9pa15/wheels/cc/34/c5/9658dea808f4b95e6bff6b1526ad8e5dd70be255e66bfa177d
Successfully built diff_gaussian_rasterization
  Running command git clone --filter=blob:none --quiet https://github.com/ashawkey/diff-gaussian-rasterization.gi

In [65]:
!/usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion Look_In_My_Eyes"

building model + flame tracking…
  warnings.warn("xFormers is available (SwiGLU)")

  warnings.warn("xFormers is available (Attention)")

  warnings.warn("xFormers is available (Block)")

INFO: using MLP layer as FFN
skip_decoder: True 
#########scale sphere:False, add_teeth:False
 Render rgb: True 
face_upsampled:(39904, 3), face_ori:torch.Size([9976, 3]),                 vertex_num_upsampled:20018, vertex_num_ori:5023
loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
finish loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
2026-06-23 10:05:22.462 | INFO     | tools.flame_tracking_single_image:__init__:69 - Output Directory: output/tracking
2026-06-23 10:05:22.462 | INFO     | tools.flame_tracking_single_image:__init__:72 - Loading Pre-trained Models...
  warnings.warn("'torch.load' received a zip file that looks like a TorchScript archive"

2026-06-23 10:05:27.505 | INFO   

In [66]:
!sed -n '40,60p' /content/LAM/tools/generateARKITGLBWithBlender.py; echo "=== template check ==="; ls -la /content/LAM/assets/sample_oac/ 2>/dev/null; echo "=== output so far ==="; ls -la /content/LAM/output/open_avatar_chat/ 2>/dev/null; find /content/LAM/output -name "*.ply" -o -name "*.glb" -o -name "*.fbx" 2>/dev/null | head

        input_mesh: Path to FLAME mesh (OBJ format)
        output_ascii_fbx: Output path for modified ASCII FBX
        template_fbx: Template FBX with FLAME structure

    Raises:
        FileNotFoundError: If input files are missing
        ValueError: If template format mismatch
    """
    logger.info(f"Updating FLAME shape in {template_fbx}")

    # Validate inputs
    if not all([input_mesh.exists(), template_fbx.exists()]):
        raise FileNotFoundError("Missing input file(s)")

    # Load and process FLAME mesh
    mesh = trimesh.load(input_mesh)
    bs_verts = np.array(mesh.vertices).flatten()
    verts_csv = ",".join([f"{v:.6f}" for v in bs_verts]) + ","

    # Read template FBX
    with template_fbx.open('r',encoding='utf-8') as f:
=== template check ===
=== output so far ===
total 12
drwxr-xr-x 3 root root 4096 Jun 23 10:06 .
drwxr-xr-x 5 root root 4096 Jun 23 10:06 ..
drwxr-xr-x 2 root root 4096 Jun 23 10:06 fisherman
/content/LAM/output/open_avatar_chat/fisherman/offse

In [67]:
!cd /content/LAM && wget -q https://virutalbuy-public.oss-cn-hangzhou.aliyuncs.com/share/aigc3d/data/LAM/sample_oac.tar && tar -xf sample_oac.tar -C assets/ && echo "--- extracted, checking ---" && ls -la assets/sample_oac/

--- extracted, checking ---
total 24224
drwxrwxrwx 2 1000  1000     4096 Apr 28  2025 .
drwxr-xr-x 6  502 staff     4096 Jun 23 10:07 ..
-rwxrwxrwx 1 1000  1000  2221524 Mar 14  2025 animation.glb
-rwxrwxrwx 1 1000  1000 22572538 Apr 28  2025 template_file.fbx


In [68]:
!/usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion Look_In_My_Eyes"

building model + flame tracking…
  warnings.warn("xFormers is available (SwiGLU)")

  warnings.warn("xFormers is available (Attention)")

  warnings.warn("xFormers is available (Block)")

INFO: using MLP layer as FFN
skip_decoder: True 
#########scale sphere:False, add_teeth:False
 Render rgb: True 
face_upsampled:(39904, 3), face_ori:torch.Size([9976, 3]),                 vertex_num_upsampled:20018, vertex_num_ori:5023
loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
finish loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
2026-06-23 10:08:43.289 | INFO     | tools.flame_tracking_single_image:__init__:69 - Output Directory: output/tracking
2026-06-23 10:08:43.290 | INFO     | tools.flame_tracking_single_image:__init__:72 - Loading Pre-trained Models...
  warnings.warn("'torch.load' received a zip file that looks like a TorchScript archive"

2026-06-23 10:08:49.613 | INFO   

In [69]:
from google.colab import files
files.download('/content/LAM/output/open_avatar_chat/fisherman.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [70]:
from google.colab import files
import shutil, os
up = files.upload()
src = list(up.keys())[0]
os.makedirs('/content/LAM/assets/sample_input', exist_ok=True)
shutil.move(src, '/content/LAM/assets/sample_input/ints.jpg')
print('saved ints.jpg')

Saving IMG_0305.JPG to IMG_0305.JPG
saved ints.jpg


In [71]:
!/usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/ints.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion Look_In_My_Eyes"

building model + flame tracking…
  warnings.warn("xFormers is available (SwiGLU)")

  warnings.warn("xFormers is available (Attention)")

  warnings.warn("xFormers is available (Block)")

INFO: using MLP layer as FFN
skip_decoder: True 
#########scale sphere:False, add_teeth:False
 Render rgb: True 
face_upsampled:(39904, 3), face_ori:torch.Size([9976, 3]),                 vertex_num_upsampled:20018, vertex_num_ori:5023
loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
finish loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
2026-06-23 10:40:51.535 | INFO     | tools.flame_tracking_single_image:__init__:69 - Output Directory: output/tracking
2026-06-23 10:40:51.535 | INFO     | tools.flame_tracking_single_image:__init__:72 - Loading Pre-trained Models...
  warnings.warn("'torch.load' received a zip file that looks like a TorchScript archive"

2026-06-23 10:40:57.632 | INFO   

In [72]:
from google.colab import files
files.download('/content/LAM/output/open_avatar_chat/ints.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
# Cell 6 (run).  SUCCESS: '\u2705 OAC zip ready: .../output/open_avatar_chat/fisherman.zip'.
!{RUN} python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path {BLENDER} --motion auto


Traceback (most recent call last):
  File "/content/LAM/colab_bake_oac.py", line 179, in <module>
    main()
  File "/content/LAM/colab_bake_oac.py", line 69, in main
    import app_lam  # importing does NOT launch gradio (that's under __main__)
  File "/content/LAM/app_lam.py", line 30, in <module>
    import moviepy.editor as mpy
  File "/usr/local/miniforge3/envs/lam/lib/python3.10/site-packages/moviepy/editor.py", line 60, in <module>
    from .video.io.sliders import sliders
  File "/usr/local/miniforge3/envs/lam/lib/python3.10/site-packages/moviepy/video/io/sliders.py", line 1, in <module>
    import matplotlib.pyplot as plt
  File "/usr/local/miniforge3/envs/lam/lib/python3.10/site-packages/matplotlib/__init__.py", line 1299, in <module>
    rcParams['backend'] = os.environ.get('MPLBACKEND')
  File "/usr/local/miniforge3/envs/lam/lib/python3.10/site-packages/matplotlib/__init__.py", line 774, in __setitem__
    raise ValueError(f"Key {key}: {ve}") from None
ValueError: Key backe

### Cell 6 — Gradio fallback (only if the headless runner errors)
Runs LAM's exact UI code in the env. Open the printed public URL → upload your image → pick a driving video
example → **tick 'Export ZIP file for Chatting Avatar'** → Generate. Zip lands in `output/open_avatar_chat/`.


In [ ]:
# Optional fallback — LAM's gradio app with a public share link, in the env.
# !sed -i 's/demo.launch()/demo.launch(share=True)/' app_lam.py
# !{RUN} python app_lam.py --blender_path {BLENDER}


In [ ]:
# Cell 7 — download the OAC zip to your Mac (Colab kernel).  SUCCESS: browser download of the zip.
from google.colab import files
import glob, os
zips = sorted(glob.glob('output/open_avatar_chat/*.zip'), key=os.path.getmtime)
assert zips, 'no OAC zip found — Cell 6 did not complete.'
print('downloading', zips[-1])
files.download(zips[-1])


In [73]:
!cd /content/LAM && wget -q -O colab_bake_oac.py https://raw.githubusercontent.com/IntsI/ints-head-gs/main/tools/colab_bake_oac.py && echo "runner updated"

runner updated


In [74]:
from google.colab import files
import shutil, os
up = files.upload()
src = list(up.keys())[0]
os.makedirs('/content/LAM/assets/sample_input', exist_ok=True)
shutil.move(src, '/content/LAM/assets/sample_input/ints_2.jpg')
print('saved ints_2.jpg')


Saving inst_2.png to inst_2.png
saved ints_2.jpg


In [75]:
!/usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/ints_2.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion Pen_Pineapple_Apple_Pen"

building model + flame tracking…
  warnings.warn("xFormers is available (SwiGLU)")

  warnings.warn("xFormers is available (Attention)")

  warnings.warn("xFormers is available (Block)")

INFO: using MLP layer as FFN
skip_decoder: True 
#########scale sphere:False, add_teeth:False
 Render rgb: True 
face_upsampled:(39904, 3), face_ori:torch.Size([9976, 3]),                 vertex_num_upsampled:20018, vertex_num_ori:5023
loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
finish loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
2026-06-23 11:38:48.881 | INFO     | tools.flame_tracking_single_image:__init__:69 - Output Directory: output/tracking
2026-06-23 11:38:48.881 | INFO     | tools.flame_tracking_single_image:__init__:72 - Loading Pre-trained Models...
  warnings.warn("'torch.load' received a zip file that looks like a TorchScript archive"

2026-06-23 11:38:55.699 | INFO   

In [76]:
from google.colab import files
files.download('/content/LAM/output/open_avatar_chat/ints_2.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [77]:
from google.colab import files
import shutil, os
up = files.upload()
src = list(up.keys())[0]
os.makedirs('/content/LAM/assets/sample_input', exist_ok=True)
shutil.move(src, '/content/LAM/assets/sample_input/ints_2.jpg')
print('saved ints_3.jpg')

Saving ints_3.png to ints_3.png
saved ints_3.jpg


In [78]:
!/usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/ints_3.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion Pen_Pineapple_Apple_Pen"

Traceback (most recent call last):
  File "/content/LAM/colab_bake_oac.py", line 191, in <module>
    main()
  File "/content/LAM/colab_bake_oac.py", line 56, in main
    assert os.path.exists(args.image), f"image not found: {args.image}"
AssertionError: image not found: assets/sample_input/ints_3.jpg
ERROR conda.cli.main_run:execute(148): `conda run bash -c cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/ints_3.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion Pen_Pineapple_Apple_Pen` failed. (See above for error)


In [79]:
from google.colab import files
import shutil, os
up = files.upload()
src = list(up.keys())[0]
os.makedirs('/content/LAM/assets/sample_input', exist_ok=True)
shutil.move(src, '/content/LAM/assets/sample_input/ints_3.jpg')
print('saved ints_3.jpg')

Saving ints_3.png to ints_3.png
saved ints_3.jpg


In [80]:
!/usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/ints_3.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion Pen_Pineapple_Apple_Pen"

building model + flame tracking…
  warnings.warn("xFormers is available (SwiGLU)")

  warnings.warn("xFormers is available (Attention)")

  warnings.warn("xFormers is available (Block)")

INFO: using MLP layer as FFN
skip_decoder: True 
#########scale sphere:False, add_teeth:False
 Render rgb: True 
face_upsampled:(39904, 3), face_ori:torch.Size([9976, 3]),                 vertex_num_upsampled:20018, vertex_num_ori:5023
loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
finish loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
2026-06-23 11:56:09.759 | INFO     | tools.flame_tracking_single_image:__init__:69 - Output Directory: output/tracking
2026-06-23 11:56:09.760 | INFO     | tools.flame_tracking_single_image:__init__:72 - Loading Pre-trained Models...
  warnings.warn("'torch.load' received a zip file that looks like a TorchScript archive"

2026-06-23 11:56:15.393 | INFO   

In [81]:
from google.colab import files
files.download('/content/LAM/output/open_avatar_chat/ints_3.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [82]:
from google.colab import files
import shutil, os
up = files.upload()
src = list(up.keys())[0]
os.makedirs('/content/LAM/assets/sample_input', exist_ok=True)
shutil.move(src, '/content/LAM/assets/sample_input/ints_4.jpg')
print('saved ints_4.jpg')

Saving ints_4.png to ints_4.png
saved ints_4.jpg


In [83]:
!/usr/local/miniforge3/bin/conda run -n lam --no-capture-output bash -c "cd /content/LAM && MPLBACKEND=Agg python colab_bake_oac.py --image assets/sample_input/ints_4.jpg --blender_path /content/blender-4.0.2-linux-x64/blender --motion Pen_Pineapple_Apple_Pen"

building model + flame tracking…
  warnings.warn("xFormers is available (SwiGLU)")

  warnings.warn("xFormers is available (Attention)")

  warnings.warn("xFormers is available (Block)")

INFO: using MLP layer as FFN
skip_decoder: True 
#########scale sphere:False, add_teeth:False
 Render rgb: True 
face_upsampled:(39904, 3), face_ori:torch.Size([9976, 3]),                 vertex_num_upsampled:20018, vertex_num_ori:5023
loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
finish loading pretrained weight from: ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors
2026-06-23 12:07:13.009 | INFO     | tools.flame_tracking_single_image:__init__:69 - Output Directory: output/tracking
2026-06-23 12:07:13.009 | INFO     | tools.flame_tracking_single_image:__init__:72 - Loading Pre-trained Models...
  warnings.warn("'torch.load' received a zip file that looks like a TorchScript archive"

2026-06-23 12:07:18.455 | INFO   

In [84]:
from google.colab import files
files.download('/content/LAM/output/open_avatar_chat/ints_4.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
### Verify before baking a batch
Drop the downloaded zip into `ints-head-gs`: `http://localhost:5173/?avatar=<url>` or the drag-drop zone
(validates the 4 files + folder-name rule per `docs/AVATAR_FORMAT.md`). Confirm **one** avatar renders first.
